<a href="https://colab.research.google.com/github/Ethan-Brooke/APF-Paper-8-Admissibility-Capacity-Ledger/blob/main/APF_Reviewer_Walkthrough.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Paper 8 — The Admissibility-Capacity Ledger
## Reviewer Walkthrough · Phase 22 Edition

[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.18604548.svg)](https://doi.org/10.5281/zenodo.18604548)

**Author:** E.S. Brooke · **Paper:** v2.9 main + v2.5 supplement (2026-04-24) · **Codebase:** v6.9

---

### What this notebook is

An interactive walkthrough of the Paper 8 argument. Every claim in the paper maps to a **check function** in the `apf/` codebase. This notebook runs each check in dependency order and renders its mathematical content as typeset equations alongside the code.

### What this notebook is **not**

A re-derivation of the entire APF framework. Paper 8 *imports* the gauge group, the SM field content, the T12 partition, and $d_{\rm eff} = 102$ from Papers 1–6. It formalises their **joint consistency** at the ledger level and proves one structurally loaded theorem (Theorem 1.1, the gauge-cosmological bridge). The rest is consistency machinery.

### Before you begin

If you are a cold AI agent or a human reviewer new to APF, read these three files in the `ai_context/` directory **first**:

1. **`ARGUMENT_FLOW.md`** — the one-page spine.
2. **`LOCAL_VS_IMPORTED.md`** — what this paper proves vs imports.
3. **`DO_NOT_CLAIM.md`** — predictable overclaims and how to avoid them.

Then run `minimal_working_example/toy_interface_numpy.py` to confirm the ledger machinery works at $K=3$, $d_{\rm eff}=4$ on your machine.

### Reading order for this notebook

$$
\underbrace{\text{§1 Setup}}_{\text{clone + imports}}
\;\longrightarrow\;
\underbrace{\text{§2 The ACC Record}}_{(K,\,\mathrm{ACC})}
\;\longrightarrow\;
\underbrace{\text{§3 Six Projections}}_{\pi_T,\pi_G,\pi_Q,\pi_F,\pi_C,\pi_A}
\;\longrightarrow\;
\underbrace{\text{§4 Four Identities}}_{I_1,\,\mathbf{I_2},\,I_3,\,I_4}
\;\longrightarrow\;
\underbrace{\text{§5 The Formal Kernel}}_{\text{Theorem 1.1}}
\;\longrightarrow\;
\underbrace{\text{§6 Predictions}}_{\Omega_\Lambda=42/61,\;\rho_\Lambda,\;H_0}
\;\longrightarrow\;
\underbrace{\text{§7 MWE}}_{K=3\text{ toy}}
$$

**$I_2$ is bold because it is the only nontrivial identity.** Everything else is modest consistency machinery. This hierarchy is the paper's own framing — see Supplement §1.6.


## §1 · Setup

Clone the paper-companion repo, install the bundled `apf/` package, and load the rendering helpers. The bundled codebase is a task-scoped subset of APF v6.9; it contains exactly what is needed to verify Paper 8's claims.

In [ ]:
# Clone the repo (no-op if already cloned)
!git clone -q https://github.com/Ethan-Brooke/APF-Paper-8-Admissibility-Capacity-Ledger.git 2>/dev/null || true
%cd -q APF-Paper-8-Admissibility-Capacity-Ledger
!pip install -q -e . 2>&1 | tail -3

In [ ]:
from apf.bank import REGISTRY, get_check, run_all
from apf.apf_utils import dag_reset, dag_put, dag_has, dag_get
from fractions import Fraction
from IPython.display import display, Markdown, Latex, HTML
import inspect

print(f'Bank registry loaded: {len(REGISTRY)} checks across modules.')
print(f'Paper 8 local checks: ~18 structural identities + provenance + predictions.')

### The `show()` helper — Phase 22 gorgeous-math edition

Every check returns a result dict with typed fields (`consistent`, `key_result`, `witnesses`, `dependencies`, epistemic tag, etc.). The `show()` helper below:

1. **Runs the check** and reports the pass/fail flag as a coloured badge.
2. **Renders the statement** from the check's docstring as typeset text.
3. **Surfaces the mathematical content** — any field containing a formula gets LaTeX-rendered; any numerical prediction gets pretty-printed with uncertainty.
4. **Lists dependencies** as a derivation-chain bullet list.
5. **Shows the epistemic tag** with an explanatory colour code.

Colour code:
- 🟢 **`[P]`** — proved from A1.
- 🟡 **`[P_structural]`** — conditional on an upstream derivation chain.
- 🟣 **`[P_arith]`** — arithmetic identity once the formula is chosen.
- 🔵 **`[P+lattice]`** — proved using lattice QCD input.
- 🟠 **`[C]`** — conjecture; open, flagged in the paper.
- 🔴 **`[FAIL]`** — check did not pass; this is the bug report.

In [ ]:

def _epistemic_badge(tag):
    """Colour-coded epistemic status badge."""
    colors = {
        'P': ('🟢', '#2ecc71', 'proved from A1'),
        'P_structural': ('🟡', '#f1c40f', 'conditional on upstream derivation'),
        'P_arith': ('🟣', '#9b59b6', 'arithmetic identity once formula chosen'),
        'P+lattice': ('🔵', '#3498db', 'proved using lattice QCD input'),
        'C': ('🟠', '#e67e22', 'conjecture; open, flagged'),
    }
    emoji, col, explain = colors.get(str(tag).strip('[]'), ('⚪', '#7f8c8d', 'unknown'))
    return f'<span style="background:{col}22;border:1px solid {col};border-radius:4px;padding:2px 8px;color:{col};font-weight:600">{emoji} {tag}</span> <em style="color:#666;font-size:0.9em">{explain}</em>'


def _render_value(v, key=''):
    """Pretty-print a value from a check result dict."""
    if isinstance(v, Fraction):
        return f'$\\displaystyle \\frac{{{v.numerator}}}{{{v.denominator}}} = {float(v):.6f}$'
    if isinstance(v, dict) and all(isinstance(val, Fraction) for val in v.values()):
        items = [f'{k}={Fraction(frac).numerator}/{Fraction(frac).denominator}' for k, frac in v.items()]
        return '$' + ',\\ '.join(items) + '$'
    if isinstance(v, float):
        return f'`{v:.9g}`'
    if isinstance(v, str) and ('=' in v or '\\\\' in v):
        # Looks like math — try LaTeX rendering
        return f'${v}$' if '$' not in v else v
    if isinstance(v, (list, tuple)) and len(v) < 8:
        return '`' + ', '.join(str(x) for x in v) + '`'
    return f'`{v}`'


def show(check_name, *, run=True, verbose=True):
    """Gorgeous-math rendering of a bank check.

    Displays the check's statement, runs it, reports pass/fail as a
    colour-coded badge, and surfaces every mathematical field in the
    result dict with LaTeX where applicable.
    """
    try:
        check = get_check(check_name)
    except KeyError:
        display(Markdown(f'**❓ Check `{check_name}` not found in registry.**'))
        return None

    # Header
    name = check_name.replace('check_', '')
    display(Markdown(f'#### `{check_name}`'))

    # Docstring statement
    doc = (check.__doc__ or '').strip()
    first_line = doc.split('\n')[0] if doc else '(no docstring)'
    display(Markdown(f'**Statement:** {first_line}'))

    if not run:
        return None

    # Run the check
    try:
        result = check()
        passed = True
    except Exception as e:
        result = {'error': f'{type(e).__name__}: {e}'}
        passed = False

    # Epistemic tag from result dict if present
    tag = 'P'
    if isinstance(result, dict):
        for k in ('epistemic_status', 'epistemic', 'tag'):
            if k in result:
                tag = result[k]
                break
    if not passed:
        tag = 'FAIL'
        display(Markdown(f'**Status:** <span style="color:#e74c3c;font-weight:700">🔴 [FAIL]</span>'))
    else:
        display(Markdown(f'**Status:** {_epistemic_badge(tag)}'))

    # Key result / witness
    if isinstance(result, dict):
        if 'key_result' in result:
            display(Markdown(f'**Key result:** {_render_value(result["key_result"])}'))
        if 'error' in result:
            display(Markdown(f'**Error:** `{result["error"]}`'))

        # Surface any other mathematical fields
        if verbose:
            skip = {'key_result', 'name', 'epistemic', 'epistemic_status', 'tag', 'dependencies',
                    'cross_refs', 'error', 'artifacts', 'statement', 'identity', 'consistent'}
            extra = {k: v for k, v in result.items() if k not in skip}
            if extra:
                rows = []
                for k, v in list(extra.items())[:10]:
                    rows.append(f'| `{k}` | {_render_value(v, k)} |')
                if rows:
                    display(Markdown('**Fields surfaced by the check:**\n\n| Field | Value |\n|---|---|\n' + '\n'.join(rows)))

        # Dependencies
        if 'dependencies' in result and result['dependencies']:
            deps = result['dependencies']
            if isinstance(deps, (list, tuple)):
                deps_str = ' · '.join(f'`{d}`' for d in deps)
                display(Markdown(f'**Depends on:** {deps_str}'))

    return result


print('show() helper loaded. Phase 22 gorgeous-math rendering enabled.')


## §2 · The ACC Record

Every admissibility interface — the Standard Model, a horizon, a quantum subsystem — is characterised in Paper 8 by a single two-scalar record:

$$\boxed{\quad \mathbf{ACC}(K,\, d_{\rm eff}) \;=\; \Big(K,\; \mathrm{ACC} \,\big)\,, \qquad \mathrm{ACC} := K\,\ln d_{\rm eff}\,. \quad}$$

- **$K \in \mathbb{Z}_{\geq 1}$** — the *structural capacity*. Integer count of admissibility channels at the interface.
- **$d_{\rm eff} > 0$** — the *per-slot effective dimension*. At interfaces where conditions SC1–SC3 hold (literal tensor-product factorisation), $d_{\rm eff}$ is a genuine Hilbert dimension. At the SM interface, SC1 fails and $d_{\rm eff} = 102$ is an **effective ledger degeneracy** (Supplement §A.2 box; see `DO_NOT_CLAIM.md` row 7).
- **$\mathrm{ACC} := K\,\ln d_{\rm eff}$** — the *log-count scalar*, equal to $\ln N$ where $N = d_{\rm eff}^K$ is the microstate count.

At the SM interface:

$$ (K_{\rm SM},\; d_{\rm eff}^{\rm SM}) \;=\; (61,\; 102)\,, \qquad N_{\rm SM} = 102^{61},\qquad \mathrm{ACC}_{\rm SM} = 61 \cdot \ln 102 \approx 282.12\,. $$

Both $K = 61$ and $d_{\rm eff} = 102$ are **imported** into Paper 8. $K = 61$ is derived in Paper 2 / Paper 4 via $L_{\rm count}$: $45 + 4 + 12 = 61$ (fermion + Higgs + gauge slot counts). $d_{\rm eff} = 102$ is derived in Paper 6 via $T_{\rm horizon\_reciprocity} + L_{\rm self\_exclusion}$: $(61-1) + 42 = 102$.

In [ ]:
from apf.unification import ACC, acc_SM, acc_horizon, acc_quantum
from apf.gauge import check_L_count
from apf.gravity import check_L_self_exclusion

# Populate the DAG with upstream derivations (L_count → C_total=61; L_self_exclusion → d_eff=102)
dag_reset()
check_L_count()
check_L_self_exclusion()

# Now build the canonical SM ACC record — reads K and d_eff from the DAG
acc_sm = acc_SM()
print(f'ACC(SM):  K = {acc_sm.K},  d_eff = {acc_sm.d_eff},  ACC scalar = {acc_sm.value:.4f}')
print(f'          N = d_eff^K = {acc_sm.d_eff}^{acc_sm.K} ≈ {acc_sm.N:.4e}')
print(f'          partition (fermion + Higgs + gauge): {acc_sm.partition}')
print(f'          residuals (Ω_b, Ω_c, Ω_Λ): {acc_sm.residuals}')

## §3 · Six Regime Projections

The ACC record is read simultaneously in six physical regimes. Each is a structure-preserving map $\pi_X : \mathbf{ACC} \to \text{(regime-specific value)}$:

$$
\begin{aligned}
\pi_T(\mathbf{ACC}) &= K\,\ln d_{\rm eff} &&\quad \text{(thermo: log-count)} \\
\pi_G(\mathbf{ACC}) &= \text{(gauge template; undefined if trivial)} &&\quad \text{(gauge regime)} \\
\pi_Q(\mathbf{ACC}) &= d_{\rm eff}^{\,K} \;=\; N &&\quad \text{(quantum: dim $\mathcal{H}_{\rm micro}$)} \\
\pi_F(\mathbf{ACC}) &= K &&\quad \text{(field content: integer count)} \\
\pi_C(\mathbf{ACC}) &= \left\{\tfrac{K_X}{K}\right\}_X &&\quad \text{(cosmological: fractions)} \\
\pi_A(\mathbf{ACC}) &= \lim_{\beta \to 0} \ln Z(\beta) \,\big|_{\,\mathrm{interface}} \;=\; K\,\ln d_{\rm eff} &&\quad \text{(action: partition function)}
\end{aligned}
$$

**Fractional Reading Equivalence.** At any interface with a residual partition $\{K_X\}$, the three fraction-forming reads agree:

$$ \frac{K_X}{K} \;=\; \frac{K_X \ln d_{\rm eff}}{K\,\ln d_{\rm eff}} \;=\; \frac{K_X}{K}\,. $$

Slot fraction $=$ entropy fraction $=$ cosmological fraction. At the SM interface with $\{K_b, K_c, K_\Lambda\} = \{3, 16, 42\}$:

$$ \Omega_b = \tfrac{3}{61},\quad \Omega_c = \tfrac{16}{61},\quad \Omega_m = \tfrac{19}{61},\quad \Omega_\Lambda = \tfrac{42}{61}\,. $$

*Honest note.* FRE is mathematically correct but is **not** the theorem that derives $\{3, 16, 42\}$. It transports the partition across readings. See `CLAIMS_LEDGER.md` row 7 and `DO_NOT_CLAIM.md` row 9 for the provenance attack surface.

In [ ]:
from apf.unification import pi_T, pi_Q, pi_F, pi_C

display(Markdown('### SM regime projections on `acc_sm`'))
display(Markdown(
    f'**$\pi_F$** = field-content integer = **{pi_F(acc_sm)}**  \n'
    f'**$\pi_Q$** = $d_{{\rm eff}}^K$ = **{pi_Q(acc_sm):.3e}**  \n'
    f'**$\pi_T$** = $K\ln d_{{\rm eff}}$ = **{pi_T(acc_sm):.4f}**  \n'
    f'**$\pi_C$** = residual fractions = **{dict(pi_C(acc_sm))}**'
))

In [ ]:
# Verify Fractional Reading Equivalence lemmas
show('check_L_Omega_Lambda_is_entropy_fraction')

In [ ]:
show('check_L_Omega_b_is_entropy_fraction')

In [ ]:
show('check_L_Omega_c_is_entropy_fraction')

## §4 · The Four Consistency Identities

The ACC architecture is only well-defined if the six projections are pairwise consistent. Four identities enforce this. **Only $I_2$ is structurally loaded** — the other three are modest consistency theorems. This hierarchy is the paper's own framing (Supplement §1.6) and should guide every reviewer's priority:

| Identity | Statement | Status |
|---|---|---|
| **$I_1$** | $K_{\rm horizon}$ read in nats (Conv. A) $=$ $K_{\rm horizon}$ read in bits (Conv. B) at the BH-matching interface | 🟡 **modest** (definitional equivalence) |
| **$I_2$** | $\pi_F$-integer at SM $=$ $\pi_C$-denominator at SM $=$ **61** | 🔴 **nontrivial** — load-bearing |
| **$I_3$** | $S_{\rm vN}(\rho_{\max}) = \ln \dim \mathcal{H} = \mathrm{ACC}$ | 🟢 standard finite-dim |
| **$I_4$** | $\lim_{\beta \to 0} \ln Z(\beta) = K \ln d_{\rm eff}$ | 🟢 standard partition-function limit |

### The $I_2$ bridge, stated sharply

$$\boxed{\quad I_2:\;\; \pi_F(\mathbf{ACC}_{\rm SM}) \;=\; K_{\rm SM} \;=\; 61 \;=\; \text{denom}\big(\pi_C(\mathbf{ACC}_{\rm SM})\big)\,. \quad}$$

The integer $K$ that the **field-content projection** reads at the SM interface (fermion + Higgs + gauge slot count, from Paper 2's $L_{\rm count}$) is the *same integer* as the denominator under which the **cosmological projection** writes the $\Omega$ fractions (from Paper 6's $T_{11}$). That these are the same 61 is Paper 8's only structurally loaded claim.

The integer form is a corollary. The load-bearing content is **Theorem 1.1** in §5 below: there exists a unique 42-dim $G_{\rm SM}$-invariant subspace $V_\Lambda \subset V_{61}$, forcing $\Omega_\Lambda = 42/61$ at the subspace level.

In [ ]:
show('check_T_ACC_unification')

In [ ]:
show('check_T_three_level_unification')

In [ ]:
show('check_T_subspace_functors_unified')

### Phase 22 anti-smuggling: `acc_SM()` is now provenance-driven

Prior to 2026-04-24, `acc_SM()` installed the residual partition as hardcoded constants:

```python
residuals={'Omega_b': _CANON_OMEGA_B_NUM,    # = 3
           'Omega_c': _CANON_OMEGA_C_NUM,    # = 16
           'Omega_Lambda': _CANON_OMEGA_LAMBDA_NUM}  # = 42
```

That meant the $I_2$ check verified $3 + 16 + 42 = 61$ **on numbers installed at record-construction time** — a consistency check, not a derivation. The **Phase 22** fix reads these from the DAG when upstream `L_equip` (in `apf/cosmology.py`) has run:

```python
Omega_b_num = (dag_get('n_baryon', ...) if dag_has('n_baryon')
               else _CANON_OMEGA_B_NUM)  # CANON = bootstrap fallback only
```

The anti-smuggling tests in `apf/test_no_smuggling.py` prove: (i) `acc_SM()` honours upstream DAG mutations, (ii) $I_2$ fails on sum-violating mutations (not a rubber stamp), (iii) CANON constants agree with `L_equip`'s derivation at v6.9 (no silent drift).

In [ ]:
# Run the Phase 22 anti-smuggling test suite
from apf.test_no_smuggling import run_all_smuggling_tests
results = run_all_smuggling_tests()
for name, res in results.items():
    emoji = '✅' if res['status'] == 'PASS' else '❌'
    claim = res.get('claim', res.get('error', ''))
    display(Markdown(f'{emoji} **`{name}`** — {claim}'))

## §5 · The Formal Kernel — Theorem 1.1

The paper's only nontrivial theorem. Everything else is either imported, arithmetic, or consistency machinery. **If you read only one thing in this notebook, read this section.**

### Setup (Supplement §1.1–1.3, stated in conventional representation-theoretic language)

**Definition (V_{61}).** Let $V_{61}$ be the 61-dimensional complex vector space with the ordered basis:

$$\{e_i^{(\rm gauge)}\}_{i=1}^{12} \;\cup\; \{e_j^{(\rm Higgs)}\}_{j=1}^{4} \;\cup\; \{e_k^{(\rm fermion)}\}_{k=1}^{45}\,, \qquad 12 + 4 + 45 = 61\,.$$

**Definition ($G_{\rm SM}$-action).** Let $G_{\rm SM} := SU(3) \times SU(2) \times U(1)$. Define the representation $\rho_{\rm SM}: G_{\rm SM} \to GL(V_{61})$ by:

$$\rho_{\rm SM} \;=\; \underbrace{\rho_{\rm gauge}}_{\dim \,12 \;=\; 8 \,+\, 3 \,+\, 1} \;\oplus\; \underbrace{\rho_{\rm Higgs}}_{\dim \,4 \;=\; \text{complex doublet}_{Y=1/2}} \;\oplus\; \underbrace{\rho_{\rm fermion}}_{\dim \,45 \;=\; 3 \,\text{gens}\, \times\, 15 \,\text{Weyl}}\,.$$

**Definition (T12 partition).** Paper 6 establishes the interface partition:

$$V_{61} \;=\; V_{\rm local} \,\oplus\, V_\Lambda\,, \qquad \dim V_{\rm local} = 19,\qquad V_{\rm local} \;=\; V_b \,\oplus\, V_c,\qquad (\dim V_b,\dim V_c) = (3, 16)\,.$$

### The theorem

$$\boxed{
\begin{aligned}
\textbf{Theorem 1.1 (Gauge-cosmological bridge).}\; & \text{With } V_{61},\, \rho_{\rm SM},\, V_{\rm local} \text{ as above:} \\
\text{(i)}\;\;& \exists\; V_\Lambda \subset V_{61} \text{ with } \dim V_\Lambda = 42,\, \rho_{\rm SM}\text{-invariant},\, V_{61} = V_{\rm local} \oplus V_\Lambda\,. \\
\text{(ii)}\;\;& \text{Such } V_\Lambda \text{ is unique.} \\
\text{(iii)}\;\;& V_{61} = V_b \oplus V_c \oplus V_\Lambda \;\text{ with }\; 3 + 16 + 42 = 61\,. \\
\text{(iv)}\;\;& \Omega_\Lambda \;=\; \dfrac{\dim V_\Lambda}{\dim V_{61}} \;=\; \dfrac{42}{61}\,. \\
\end{aligned}
}$$

### Proof sketch (Supplement §1.5, four steps)

**Step 1.** By Maschke's theorem for compact Lie groups (Hall 2015, Thm. 4.28), every finite-dimensional representation of $G_{\rm SM}$ over $\mathbb{C}$ is completely reducible. In particular, the $\rho_{\rm SM}$-invariant subspace $V_{\rm local}$ has a $\rho_{\rm SM}$-invariant complement.

**Step 2.** Any $\rho_{\rm SM}$-invariant complement of $V_{\rm local}$ must have dimension $61 - 19 = 42$.

**Step 3.** By semisimplicity and the irrep-content specification of $V_{\rm local}$, any two $G_{\rm SM}$-invariant complements of $V_{\rm local}$ are isomorphic; and since $V_{\rm local}$'s isotypic component completion in $V_{61}$ is unique, the complement is unique as a subspace.

**Step 4.** Define $V_\Lambda$ as this unique complement; (iii) and (iv) follow by dimension counting. $\qquad \blacksquare$

### Honest uncertainty

Theorem 1.1 forces uniqueness of $V_\Lambda$ **given the irrep-level specification of $V_{\rm local}$**. If $V_{\rm local}$ is defined set-theoretically ("span of matter slots") rather than representation-theoretically ("sum of specific irreps of $\rho_{\rm SM}$"), the proof flattens. Paper 8 imports the irrep-level specification from Paper 4 §5 — see `LOCAL_VS_IMPORTED.md` and `DO_NOT_CLAIM.md` row 9.

### Code witness

Theorem 1.1's subspace-level content is witnessed in `apf/gravity.py` by `check_T_interface_sector_bridge`, which confirms the 42-dim $V_{\rm global} \cong V_\Lambda$ identification at the DAG level:

In [ ]:
show('check_T_interface_sector_bridge')

In [ ]:
show('check_L_global_interface_is_horizon')

## §6 · Downstream Predictions

Everything in this section is a *consequence* of the structural core in §5. Numerical agreement here is evidence for the architecture only *after* the structural argument is accepted. Numerical agreement *alone* is insufficient — see `DO_NOT_CLAIM.md` row 12.

### Cosmological fractions

$$
\Omega_b = \tfrac{3}{61} \approx 0.0492\,,\qquad
\Omega_c = \tfrac{16}{61} \approx 0.2623\,,\qquad
\Omega_m = \tfrac{19}{61} \approx 0.3115\,,\qquad
\Omega_\Lambda = \tfrac{42}{61} \approx 0.6885\,.
$$

Planck 2018 posterior: $\Omega_\Lambda = 0.6847 \pm 0.0073$ (0.5$\sigma$ agreement).

### Absolute $\rho_\Lambda$

$$\boxed{\quad \frac{\rho_\Lambda}{M_{\rm Pl}^4} \;=\; \frac{42}{102^{62}} \;\approx\; 10^{-122.91}\,. \quad}$$

Observed: $\approx 10^{-122.90}$. Agreement: **3% (0.012 decades)** with **zero free parameters**. The coefficient $42/102$ is privileged **structurally**, not numerically — an exhaustive scan in `apf/lambda_absolute.py` finds 20 APF-native candidates within 0.01 decades. Status: `[C]` on the structural-coefficient derivation.

### $H_0$ from $\rho_\Lambda$

Inverting via GR $\rho_{\rm crit}$:

$$H_0 \;=\; \sqrt{\frac{8\pi \cdot 61}{3}} \,\cdot\, 102^{-31} \,\cdot\, M_{\rm Pl} \;\approx\; \mathbf{70.03\ \text{km/s/Mpc}}\,.$$

Sits at the midpoint of the Planck–SH0ES tension (67.36 vs 73.04). The 7.09$\sigma$ tension against Planck 2018 is stated as a **falsifier** in Paper 6 §11.4 — not resolved here. See `DO_NOT_CLAIM.md` row 6.

### $T_{\rm CMB}$ parallel formula

$$\left(\frac{T_{\rm CMB}}{M_{\rm Pl}}\right)^{\!4} \;=\; \frac{48}{102^{64}}\,,\qquad \text{FIRAS agreement: 0.33\%.}$$

In [ ]:
show('check_L_Lambda_absolute_numerical_formula')

In [ ]:
show('check_T_Lambda_to_H0_inversion')

In [ ]:
show('check_T_T_CMB_absolute_formula')

## §7 · Minimal Working Example

The smallest non-trivial instance of the ledger machinery. $K = 3$, $d_{\rm eff} = 4$, uniform residual partition $(1, 1, 1)$.

$$ V_3 = \mathbb{C}^3\,,\quad H_{\rm micro} = (\mathbb{C}^4)^{\otimes 3} = \mathbb{C}^{64}\,,\quad \mathrm{ACC}_{\rm toy} = 3 \ln 4 \approx 4.16\,. $$

Run the toy and verify every identity by hand.

In [ ]:
# Pull the MWE out of the repo and run it
import subprocess, sys
result = subprocess.run([sys.executable, 'minimal_working_example/toy_interface_numpy.py'],
                        capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)

### What the MWE does not prove

Running this toy confirms the ledger machinery is well-defined mathematics at small scale. It does **not** prove that the SM interface has $K = 61$, $d_{\rm eff} = 102$, or that $\Omega_\Lambda = 42/61$ is the right cosmological fraction. Those are structural identifications defended by Theorem 1.1 and the upstream derivation chain — not by this script.

See `minimal_working_example/WHY_THIS_MATTERS.md` for the pedagogical framing.

## §8 · Full bank pass

Run every check in the bundled Paper 8 codebase subset. Target: 100% pass (modulo scipy-dependent checks that require `pip install scipy`).

In [ ]:
from apf.bank import run_all
results = run_all()
from collections import Counter
outcomes = Counter()
for name, res in results.items():
    if isinstance(res, dict) and 'error' in res:
        outcomes['ERROR'] += 1
    else:
        outcomes['PASS'] += 1
print(f'Total: {len(results)} checks')
for status, n in outcomes.most_common():
    print(f'  {status}: {n}')

## Where to go next

- **Supplement §1** — Theorem 1.1 full proof + context.
- **Supplement §O** — "Gauge-Cosmological Bridge: Integrated Nontrivial Content" centerpiece section.
- **Supplement §R** — Minimal working example write-up.
- **`ai_context/CLAIMS_LEDGER.md`** — row-by-row attack surface for every load-bearing claim.
- **`ai_context/LOCAL_VS_IMPORTED.md`** — local-vs-imported partition.
- **`ai_context/DO_NOT_CLAIM.md`** — 20 pre-registered anti-hallucination guards.
- **`apf/test_no_smuggling.py`** — anti-smuggling mutation test suite.

### Citation

```bibtex
@software{Brooke_Paper8_2026,
  author  = {Brooke, Ethan S.},
  title   = {The Admissibility-Capacity Ledger},
  year    = 2026,
  version = {v2.9 main + v2.5 supplement},
  doi     = {10.5281/zenodo.18604548}
}
```

*Paper-companion repo · Paper 8 v2.9/v2.5 · Phase 22 gorgeous-math edition · 2026-04-24.*